In [1]:
import pandas as pd
import re
import seaborn as sns

CLEANING DATA

In [2]:
df = pd.read_csv('data_label.csv', sep=';')

df.head()

,text,label
0,Kunjungan Prabowo ini untuk meresmikan dan men...,Sumber Daya Alam
1,RT Anies dapat tepuk tangan meriah saat jadi R...,Politik
2,@CIqXqwGAT04tMtx4OCATxjoVq7vv/Y8HeYaIOgMFg8Y= ...,Demografi
3,RT @L3R8XFBw3WGbxRPSj0/0hHZTbqVGX7qtfwRg9zmhK7...,Politik
4,Anies Baswedan Harap ASN termasuk TNI dan Polr...,Politik


In [3]:
# cek duplikasi
df.shape

(5000, 2)

In [4]:
# drop semua yg duplikat
df = df.drop_duplicates(subset=['text'])
df.duplicated().sum()

np.int64(0)

In [5]:
df = df.dropna()

In [6]:
df.isnull().sum()

text     0
label    0
dtype: int64

delete noise

In [7]:
def clean_twitter_text(text):
    text = re.sub(r'@[A-Za-z0-9_]+', '', text) #hapus mention
    text = re.sub(r'#\w+', '', text) #hapus hashtag
    text = re.sub(r'https?://\S+', '', text) #hapus URL
    text = re.sub(r'[^A-Za-z0-9\s]+', '', text) #hapus karakter khusus
    text = re.sub(r'\s+', ' ', text).strip() #hapus spasi berlebih
    text = re.sub(r'RT[\s]+', '', text) #hapus RT
    text = re.sub(r'RE[\s]+', '', text) #hapus RE
    return text

df['text'] = df['text'].apply(clean_twitter_text)


In [8]:
df['text'] = df['text'].str.lower()

In [9]:
df

,text,label
0,kunjungan prabowo ini untuk meresmikan dan men...,Sumber Daya Alam
1,anies dapat tepuk tangan meriah saat jadi rekt...,Politik
2,y8heyaiogmfg8y emng bener sih pendukung 01 ada...,Demografi
3,0hhztbqvgx7qtfwrg9zmhk7q sewaktu anies bersika...,Politik
4,anies baswedan harap asn termasuk tni dan polr...,Politik
...,...,...
4995,ngeliat debat kemaren pas prabowo kicep kekira...,Politik
4996,masyarakat yakin bahwa prabowogibran memiliki ...,Politik
4997,imo both are irrational but yg satu jauh lebih...,Ekonomi
4998,look at that pak ganjar anda sdh berkecimpung ...,Pertahanan dan Keamanan


load slang dict

In [10]:
try:
    slang_df   = pd.read_csv('slang_indo.csv', header=None, names=['slang','formal'])
    slang_dict = dict(zip(slang_df['slang'], slang_df['formal']))
    print(f'Slang loaded: {len(slang_dict)} entri')
    print(f'Contoh: {list(slang_dict.items())[:5]}')
except FileNotFoundError:
    slang_dict = {}
    print('File slang tidak ditemukan, lanjut tanpa normalisasi slang')

Slang loaded: 1250 entri
Contoh: [('aamiin', 'amin '), ('adek', 'adik '), ('adlh', 'adalah '), ('aer', 'air '), ('aiskrim', 'es krim ')]


prepo, init stemmer + stopword